# Comparison

## 1. Setup

> Which model's outputs are we evaluating, and how many of them are valid?

Loads the original dataset and the model's synthetic output, then checks the debug file to count how many records failed to generate, giving an immediate sense of generation reliability before any quality analysis.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG_FILE = "qwen7b_lora_10.json"
# ──────────────────────────────────────────────────────────────────────────────

import json, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")

config       = json.loads(Path("../configs/"+CONFIG_FILE).read_text(encoding="utf-8"))

MODEL_NAME   = Path(CONFIG_FILE).stem
BASE_DIR     = Path(config["output_dir"])
ORIG_PATH    = Path(config["dataset_dir"]) / config["original_dataset"]
SYNT_PATH  = BASE_DIR / f"{MODEL_NAME}_{config['seed']}.json"
DEBUG_PATH = BASE_DIR / f"{MODEL_NAME}_{config['seed']}_debug.json"

num_cols      = config["num_cols"]
cat_cols      = config["cat_cols"]
nn_cols       = config["nn_cols"]

label_cfg     = config.get("label_analysis",   {"enabled": False})
geo_cfg       = config.get("geo_analysis",      {"enabled": False})
temporal_cfg  = config.get("temporal_analysis", {"enabled": False})

with open(ORIG_PATH) as f:
    orig = pd.DataFrame(json.load(f))

if SYNT_PATH.exists():
    with open(SYNT_PATH) as f:
        synt = pd.DataFrame(json.load(f))
else:
    synt = pd.DataFrame()

if DEBUG_PATH.exists():
    with open(DEBUG_PATH) as f:
        debug_raw = json.load(f)
    debug_entries = debug_raw if isinstance(debug_raw, list) else debug_raw.get("errors", [])
    n_errors = len(debug_entries)
else:
    debug_entries = []
    n_errors = 0

In [ ]:
n_valid = len(synt)
n_total = n_valid + n_errors
pct_err = n_errors / n_total * 100 if n_total > 0 else 0

print(f"Model  : {MODEL_NAME}")
print(f"Valids   : {n_valid:>6,}  ({100 - pct_err:.2f}%)")
print(f"Errors   : {n_errors:>6,}  ({pct_err:.2f}%)")
print(f"Total   : {n_total:>6,}")

if not n_valid:
    raise FileNotFoundError("No valid synthetic data generated. Stopping comparison.")

orig["source"] = "original"
synt["source"] = "synthetic"
combined = pd.concat([orig, synt], ignore_index=True)

## 2. Data quality

> Are the synthetic and original datasets structurally identical?

A basic null-check and shape inspection on both DataFrames, plus a descriptive statistics comparison.

In [ ]:
print("=== ORIGINAL ===")
print(orig.shape)
print(orig.isnull().sum()[orig.isnull().sum() > 0])

print("\n=== SYNTHETIC ===")
print(synt.shape)
print(synt.isnull().sum()[synt.isnull().sum() > 0])

In [ ]:
desc_orig = orig[num_cols].describe().T
desc_synt = synt[num_cols].describe().T
desc_orig.columns = ["count_o","mean_o","std_o","min_o","25_o","50_o","75_o","max_o"]
desc_synt.columns = ["count_s","mean_s","std_s","min_s","25_s","50_s","75_s","max_s"]
pd.concat([desc_orig, desc_synt], axis=1)

## 3. Numerical Distributions

> Do the synthetic numerical features visually resemble the originals?

Side-by-side histograms and boxplots for each numerical column. Histograms show shape, skew and modes; boxplots expose differences in median, spread and outliers.

In [ ]:
fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, len(num_cols) * 3))

for i, col in enumerate(num_cols):
    for j, (df, label) in enumerate([(orig, "Original"), (synt, "Synthetic")]):
        axes[i, j].hist(df[col].dropna(), bins=60, color="steelblue" if j == 0 else "salmon", edgecolor="none", density=True)
        axes[i, j].set_title(f"{col} - {label}")
        axes[i, j].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))

for i, col in enumerate(num_cols):
    axes[i].boxplot(
        [orig[col].dropna(), synt[col].dropna()],
        tick_labels=["Original", "Synthetic"],
        patch_artist=True,
        boxprops=dict(facecolor="steelblue"),
    )
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

## 4. KS Test (Numerical)

> How statistically similar are the synthetic and original numerical distributions?

Two metrics per feature. TVD bins both distributions and measures how much probability mass differs (below 0.05 is good, below 0.10 acceptable, above that poor). KS test compares the empirical CDFs of both samples (a high statistic and low p-value means the two distributions are significantly different). TVD tells you how much they differ; KS tells you if that difference is statistically meaningful.

In [ ]:
def tvd_numerical(s1, s2, bins=50):
    all_vals = pd.concat([s1, s2]).dropna()
    bin_edges = np.linspace(all_vals.min(), all_vals.max(), bins + 1)
    p, _ = np.histogram(s1.dropna(), bins=bin_edges, density=False)
    q, _ = np.histogram(s2.dropna(), bins=bin_edges, density=False)
    p = p / p.sum()
    q = q / q.sum()
    return round(0.5 * np.abs(p - q).sum(), 4)

num_results = []
for col in num_cols:
    tvd = tvd_numerical(orig[col], synt[col])
    ks_stat, ks_p = stats.ks_2samp(orig[col].dropna(), synt[col].dropna())
    num_results.append({
        "feature": col,
        "tvd": tvd,
        "ks_stat": round(ks_stat, 4),
        "ks_p": round(ks_p, 6),
        "quality": "good" if tvd < 0.05 else "acceptable" if tvd < 0.1 else "poor"
    })

num_df = pd.DataFrame(num_results).sort_values("tvd", ascending=False)
print(num_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["steelblue" if q == "good" else "orange" if q == "acceptable" else "salmon"
          for q in num_df["quality"]]
ax.barh(num_df["feature"], num_df["tvd"], color=colors)
ax.axvline(0.05, color="steelblue", linestyle="--", linewidth=0.8, label="good (0.05)")
ax.axvline(0.10, color="orange", linestyle="--", linewidth=0.8, label="acceptable (0.10)")
ax.set_xlabel("Total Variation Distance")
ax.set_title("Numerical features - distributional similarity (TVD)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Categorical Distributions

> Do the synthetic categorical features reproduce the same category frequencies as the originals?

Relative frequencies computed via value_counts(normalize=True) and shown as grouped bar charts. Makes it immediately visible if the model over/under-represents specific categories like certain states or job types.

In [ ]:
for col in cat_cols:
    freq_orig = orig[col].value_counts(normalize=True).rename("original")
    freq_synt = synt[col].value_counts(normalize=True).rename("synthetic")
    freq = pd.concat([freq_orig, freq_synt], axis=1).fillna(0).sort_values("original", ascending=False)

    top = freq.head(20)
    top.plot(kind="bar", figsize=(14, 4), title=f"{col} - relative frequency", color=["steelblue", "salmon"])
    plt.ylabel("proportion")
    plt.tight_layout()
    plt.show()

## 6. Chi-Square Test (Categorical)

> Are the differences in categorical distributions statistically significant, and how large is the effect?

Three metrics per feature. TVD works the same as section 4 but over discrete categories. Chi-Square compares original counts against scaled synthetic counts (a significant result means the distributions are statistically different). Cramér's V is an effect size derived from chi-square, ranging from 0 to 1, and unlike p-values it doesn't inflate with sample size.

In [ ]:
def tvd_categorical(s1, s2):
    cats = set(s1.dropna().unique()) | set(s2.dropna().unique())
    p = s1.value_counts(normalize=True).reindex(cats, fill_value=0)
    q = s2.value_counts(normalize=True).reindex(cats, fill_value=0)
    return round(0.5 * np.abs(p - q).sum(), 4)

cat_results = []
for col in cat_cols:
    tvd = tvd_categorical(orig[col], synt[col])
    cats = set(orig[col].dropna().unique()) | set(synt[col].dropna().unique())
    o = orig[col].value_counts().reindex(cats, fill_value=0)
    s = synt[col].value_counts().reindex(cats, fill_value=0)
    s_scaled = (s / s.sum()) * o.sum()
    s_smoothed = s_scaled + 1e-6
    s_smoothed = s_smoothed / s_smoothed.sum() * o.sum()
    chi2, chi2_p = stats.chisquare(f_obs=o, f_exp=s_smoothed)
    n = o.sum()
    k = len(cats)
    cramers_v = round(np.sqrt(chi2 / (n * (k - 1))), 4)
    cat_results.append({
        "feature": col,
        "tvd": tvd,
        "cramers_v": cramers_v,
        "chi2_p": round(chi2_p, 6),
        "quality": "good" if tvd < 0.05 else "acceptable" if tvd < 0.1 else "poor"
    })

cat_df = pd.DataFrame(cat_results).sort_values("tvd", ascending=False)
print(cat_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
colors = ["steelblue" if q == "good" else "orange" if q == "acceptable" else "salmon"
          for q in cat_df["quality"]]
ax.barh(cat_df["feature"], cat_df["tvd"], color=colors)
ax.axvline(0.05, color="steelblue", linestyle="--", linewidth=0.8, label="good (0.05)")
ax.axvline(0.10, color="orange", linestyle="--", linewidth=0.8, label="acceptable (0.10)")
ax.set_xlabel("Total Variation Distance")
ax.set_title("Categorical features - distributional similarity (TVD)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Fraud Analysis


> Does the synthetic data preserve the fraud signal?

Three checks: class balance compares fraud vs. legitimate proportions in both datasets; amount by fraud class checks whether synthetic data learned that fraudulent transactions tend to have different amounts; fraud rate by category verifies that the model reproduced which merchant categories are more fraud-prone.

In [ ]:
if label_cfg["enabled"]:
    label_col    = label_cfg["label_col"]
    amount_col   = label_cfg["amount_col"]
    category_col = label_cfg["category_col"]

    label_orig = orig[label_col].value_counts(normalize=True)
    label_synt = synt[label_col].value_counts(normalize=True)
    label_comp = pd.DataFrame({"original": label_orig, "synthetic": label_synt})
    print(label_comp)
    label_comp.plot(kind="bar", figsize=(6,4), title="Label class balance",
                    color=["steelblue","salmon"])
    plt.ylabel("proportion"); plt.tight_layout(); plt.show()

In [ ]:
if label_cfg["enabled"]:
    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    for ax, (d, lbl) in zip(axes, [(orig,"Original"),(synt,"Synthetic")]):
        for val, color in enumerate(["steelblue","salmon"]):
            subset = d[d[label_col] == val][amount_col].dropna()
            if not subset.empty:
                ax.hist(subset, bins=60, alpha=0.6, density=True,
                        color=color, label=f"{label_col}={val}")
        ax.set_title(f"{amount_col} by {label_col} - {lbl}"); ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
if label_cfg["enabled"]:
    label_cat_orig = orig.groupby(category_col)[label_col].mean().rename("original")
    label_cat_synt = synt.groupby(category_col)[label_col].mean().rename("synthetic")
    label_cat = pd.concat([label_cat_orig, label_cat_synt], axis=1)\
                  .fillna(0).sort_values("original", ascending=False)
    label_cat.plot(kind="bar", figsize=(14,5),
                   title=f"{label_col} rate by {category_col}",
                   color=["steelblue","salmon"])
    plt.ylabel(f"{label_col} rate"); plt.tight_layout(); plt.show()

## 8. Correlation Heatmaps

> Does the synthetic data preserve the relationships between numerical features?

Pearson correlation matrices for both datasets, plus a third heatmap showing the absolute difference cell by cell. This checks whether the model learned inter-feature dependencies (not just individual distributions in isolation).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

extra = [label_cfg["label_col"]] if label_cfg["enabled"] else []
corr_cols = num_cols + extra

for ax, (df, label) in zip(axes, [(orig, "Original"), (synt, "Synthetic")]):
    corr = df[corr_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True, linewidths=0.5)
    ax.set_title(f"Correlation - {label}")

plt.tight_layout()
plt.show()

In [ ]:
extra = [label_cfg["label_col"]] if label_cfg["enabled"] else []
corr_cols = num_cols + extra

corr_orig = orig[corr_cols].corr()
corr_synt = synt[corr_cols].corr()
corr_diff = (corr_orig - corr_synt).abs()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_diff, annot=True, fmt=".2f", cmap="Oranges", square=True, linewidths=0.5)
plt.title("Correlation absolute difference (orig - synt)")
plt.tight_layout()
plt.show()

## 9. Geographic Distribution

> Does the synthetic data reproduce realistic spatial patterns?

Two checks: a scatter plot verifies that synthetic transactions fall within plausible geographic boundaries and not at invalid coordinates. The customer-to-merchant distance compares how far apart customers and merchants are in both datasets, checking whether the model preserved that realistic proximity relationship.

In [ ]:
if geo_cfg["enabled"]:
    lat, lon   = geo_cfg["lat"],      geo_cfg["lon"]
    mlat, mlon = geo_cfg["merch_lat"], geo_cfg["merch_lon"]

    fig, axes = plt.subplots(1, 2, figsize=(16,5))
    for ax, (d, lbl) in zip(axes, [(orig,"Original"),(synt,"Synthetic")]):
        ax.scatter(d[lon], d[lat], alpha=0.05, s=1, c="steelblue")
        ax.set_title(f"Transaction locations - {lbl}")
        ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    plt.tight_layout(); plt.show()



In [ ]:
if geo_cfg["enabled"]:
    for d, _ in [(orig,"Original"),(synt,"Synthetic")]:
        d["geo_distance"] = np.sqrt((d[lat]-d[mlat])**2 + (d[lon]-d[mlon])**2)

    fig, ax = plt.subplots(figsize=(10,4))
    ax.hist(orig["geo_distance"].dropna(), bins=60, alpha=0.6, density=True,
            color="steelblue", label="Original")
    ax.hist(synt["geo_distance"].dropna(), bins=60, alpha=0.6, density=True,
            color="salmon", label="Synthetic")
    ax.set_title("Customer-to-merchant geo distance")
    ax.set_xlabel("distance (degrees)"); ax.legend()
    plt.tight_layout(); plt.show()

## 10. Temporal Analysis

> Does the synthetic data reproduce the original temporal patterns and fraud seasonality?

Extracts hour, day of week and month from the Unix timestamp, then runs two comparisons. Transaction volume by time slot checks whether activity peaks are preserved. Fraud rate by time slot checks whether the model learned when fraud is more likely to happen (e.g. at night or on weekends).

In [ ]:
if temporal_cfg["enabled"]:
    time_col  = temporal_cfg["unix_time_col"]
    label_col = temporal_cfg["label_col"]

    for d in [orig, synt]:
        d["datetime"]  = pd.to_datetime(d[time_col], unit="s")
        d["hour"]      = d["datetime"].dt.hour
        d["dayofweek"] = d["datetime"].dt.dayofweek
        d["month"]     = d["datetime"].dt.month

In [ ]:
if temporal_cfg["enabled"]:
    fig, axes = plt.subplots(1, 3, figsize=(18,4))
    for ax, col in zip(axes, ["hour","dayofweek","month"]):
        orig[col].value_counts().sort_index().plot(ax=ax, label="Original",
                                                   color="steelblue", marker="o")
        synt[col].value_counts().sort_index().plot(ax=ax, label="Synthetic",
                                                   color="salmon", marker="o")
        ax.set_title(col); ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
if temporal_cfg["enabled"]:
    fig, axes = plt.subplots(1, 3, figsize=(18,4))
    for ax, col in zip(axes, ["hour","dayofweek","month"]):
        orig.groupby(col)[label_col].mean().plot(ax=ax, label="Original",
                                                 color="steelblue", marker="o")
        synt.groupby(col)[label_col].mean().plot(ax=ax, label="Synthetic",
                                                 color="salmon", marker="o")
        ax.set_title(f"{label_col} rate by {col}"); ax.legend()
    plt.tight_layout(); plt.show()

## 11. Privacy Check (Nearest Neighbor Distance)

> Is the synthetic data genuinely novel, or is it memorizing real records?

For each synthetic record, the nearest original record is found in a standardized feature space. A baseline is then computed by finding each original record's nearest other original (representing the natural density of the real data). The privacy ratio divides the two mean distances: a ratio near or above 1 means synthetic points are as far from real ones as real points are from each other, which is good. Well below 1 suggests memorization. Records closer than 0.01 are flagged as likely exact copies.

In [ ]:
sample_cols = nn_cols
sample_size = 2000

orig_sample = orig[sample_cols].dropna().sample(sample_size, random_state=config["dataset_seed"])
synt_sample = synt[sample_cols].dropna().sample(min(sample_size, len(synt)), random_state=config["dataset_seed"])

scaler = StandardScaler()
orig_scaled = scaler.fit_transform(orig_sample)
synt_scaled = scaler.transform(synt_sample)

nn = NearestNeighbors(n_neighbors=1).fit(orig_scaled)
distances, _ = nn.kneighbors(synt_scaled)

nn_baseline = NearestNeighbors(n_neighbors=2).fit(orig_scaled)
dist_baseline, _ = nn_baseline.kneighbors(orig_scaled)
baseline_dist = dist_baseline[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(distances.flatten(), bins=50, color="steelblue", edgecolor="none")
axes[0].set_title("Nearest neighbor distance: synthetic → original")
axes[0].set_xlabel("distance (normalized)")
axes[0].set_ylabel("count")

axes[1].hist(baseline_dist, bins=50, color="salmon", edgecolor="none")
axes[1].set_title("Nearest neighbor distance: original → original (baseline)")
axes[1].set_xlabel("distance (normalized)")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

privacy_ratio = distances.mean() / baseline_dist.mean()

print(f"Mean NN distance (synt→orig) : {distances.mean():.4f}")
print(f"Mean NN distance (orig→orig) : {baseline_dist.mean():.4f}")
print(f"Ratio (privacy score)        : {privacy_ratio:.4f}")
print(f"% exact copies               : {(distances.flatten() < 0.01).mean()*100:.2f}%")